# Archaic (Shakespeare) – 01 Data

Load the Shakespeare corpus, clean it, split it 90/10, and cut it into 256-token blocks for training.
The blocks are saved to disk and loaded by `02_train_generation.ipynb`.

## 1. Setup

In [1]:
from pathlib import Path
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer

In [2]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"
DATA_PATH  = "data.txt"                         # path to the provided Shakespeare file
OUT_DIR    = Path("../../results/archaic")      # shared with 02_train_generation
BLOCK      = 256                                # tokens per training block
SEED       = 42

OUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load and clean

In [3]:
raw = load_dataset("text", data_files={"train": DATA_PATH}, sample_by="paragraph")["train"]

# Rows start with a leading "\n" (where speaker labels were removed), so strip them,
# and drop anything empty.
ds = raw.map(lambda ex: {"text": ex["text"].strip()})
ds = ds.filter(lambda ex: len(ex["text"]) > 0)
print(len(raw), "->", len(ds), "rows")

print(ds[0]["text"][:500])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/30008 [00:00<?, ? examples/s]

Filter:   0%|          | 0/30008 [00:00<?, ? examples/s]

30008 -> 30008 rows
So shaken as we are, so wan with care,
Find we a time for frighted peace to pant,
And breathe short-winded accents of new broils
To be commenced in strands afar remote.
No more the thirsty entrance of this soil
Shall daub her lips with her own children's blood,
Nor more shall trenching war channel her fields,
Nor bruise her flowerets with the armed hoofs
Of hostile paces: those opposed eyes,
Which, like the meteors of a troubled heaven,
All of one nature, of one substance bred,
Did lately meet i


**Formatting notes**
- The provided data has **no speaker names and no stage directions**, so nothing had to be handled.
- Each row is one speech; line breaks inside a speech (verse lines) are kept as `\n`.
- Leading newlines are stripped and speeches are re-joined with `\n\n` before tokenizing.

## 3. Train / test split

In [4]:
# Contiguous 90/10 split (no shuffle), so test is unseen material.
data = ds.train_test_split(train_size=0.9, shuffle=False)
data

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 27007
    })
    test: Dataset({
        features: ['text'],
        num_rows: 3001
    })
})

## 4. Tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("eos:", tokenizer.eos_token, "| pad:", tokenizer.pad_token, "| vocab:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

eos: <|endoftext|> | pad: None | vocab: 49152


In [6]:
# Quick look at how text becomes tokens.
example = data["train"][0]["text"][:200]
ids = tokenizer(example)["input_ids"]
pieces = tokenizer.convert_ids_to_tokens(ids)
print(list(zip(ids, pieces))[:20])

print("\\n ->", tokenizer("\n")["input_ids"], "| \\n\\n ->", tokenizer("\n\n")["input_ids"])
print("round trip ok:", tokenizer.decode(ids) == example)

[(2931, 'So'), (39757, 'Ġshaken'), (347, 'Ġas'), (392, 'Ġwe'), (359, 'Ġare'), (28, ','), (588, 'Ġso'), (267, 'Ġw'), (276, 'an'), (351, 'Ġwith'), (1176, 'Ġcare'), (28, ','), (198, 'Ċ'), (11933, 'Find'), (392, 'Ġwe'), (253, 'Ġa'), (655, 'Ġtime'), (327, 'Ġfor'), (16742, 'Ġfright'), (277, 'ed')]
\n -> [198] | \n\n -> [1116]
round trip ok: True


The tokenizer is byte-level: `Ġ` is a space and `Ċ` is a newline in the raw pieces. `\n\n` is a single token, so speech boundaries are cheap.

## 5. Build 256-token blocks

In [7]:
def make_blocks(ids, block=BLOCK):
    n = len(ids) // block                      # leftover tail is dropped
    return [ids[i * block:(i + 1) * block] for i in range(n)]

def to_ds(blocks):
    # labels = copy of input_ids; the model shifts them internally
    return Dataset.from_dict({"input_ids": blocks, "labels": [b.copy() for b in blocks]})

train_ids = tokenizer("\n\n".join(data["train"]["text"]))["input_ids"]
test_ids  = tokenizer("\n\n".join(data["test"]["text"]))["input_ids"]

train_blocks, test_blocks = make_blocks(train_ids), make_blocks(test_ids)
assert all(len(b) == BLOCK for b in train_blocks + test_blocks)

lm_data = DatasetDict({
    "train": to_ds(train_blocks).shuffle(seed=SEED),   # shuffle train only
    "test":  to_ds(test_blocks),
})
print(f"train: {len(train_ids):,} tokens -> {len(train_blocks)} blocks")
print(f"test:  {len(test_ids):,} tokens -> {len(test_blocks)} blocks")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1112698 > 8192). Running this sequence through the model will result in indexing errors


train: 1,112,698 tokens -> 4346 blocks
test:  108,986 tokens -> 425 blocks


In [8]:
print(tokenizer.decode(lm_data["train"][0]["input_ids"])[:600])

m, perhaps.

Yes, in good faith.

Go, sickness as thou art!

Well, better wits have worn plain statute-caps.
But will you hear? the king is my love sworn.

And quick Biron hath plighted faith to me.

And Longaville was for my service born.

Dumain is mine, as sure as bark on tree.

Madam, and pretty mistresses, give ear:
Immediately they will again be here
In their own shapes, for it can never be
They will digest this harsh indignity.

Will they return?

They will, they will, God knows,
And leap for joy, though they are lame with blows:
Therefore change favours, and, when they repair,
Blow lik


## 6. Save

In [9]:
lm_data.save_to_disk(OUT_DIR / "blocks")
print("saved to", OUT_DIR / "blocks")

Saving the dataset (0/1 shards):   0%|          | 0/4346 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/425 [00:00<?, ? examples/s]

saved to ../../results/archaic/blocks
